In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import joblib

from sklearn.base import clone
from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    ExtraTreesClassifier,
    RandomForestClassifier,
)

from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix,
)

In [5]:
# ============================================================
# 1. Datos
# ============================================================
df = pd.read_csv("data/df_final.csv")
N_SPLITS = 5
SEED = 42

X = df.drop(
    columns=["movimiento", "repeticion_id", "ventana"]
).values

# Se usan etiquetas de texto para facilitar los reportes.
y = df["movimiento"].astype(str)
groups = df["repeticion_id"]
clases = np.sort(y.unique())

print(f"Ventanas: {len(X):,}")
print(f"Características: {X.shape[1]}")
print(f"Movimientos: {len(clases)}")
print(f"Grupos: {groups.nunique():,}")

Ventanas: 34,056
Características: 72
Movimientos: 15
Grupos: 4,257


In [8]:
modelos_busqueda = {
    "HistGradientBoosting": {
        "pipeline": Pipeline([
            ("imputacion", SimpleImputer(strategy="median")),
            ("modelo", HistGradientBoostingClassifier(
                early_stopping=False,
                random_state=SEED,
            )),
        ]),

        "parametros": {
            "modelo__learning_rate": [0.03, 0.05, 0.08, 0.1, 0.15],
            "modelo__max_iter": [100, 200, 300, 400],
            "modelo__max_leaf_nodes": [7, 15, 31, 63],
            "modelo__max_depth": [None, 5, 10, 15],
            "modelo__min_samples_leaf": [10, 20, 30, 40],
            "modelo__l2_regularization": [0, 0.1, 0.5, 1, 5],
        },
    },

    "Extra Trees": {
        "pipeline": Pipeline([
            ("imputacion", SimpleImputer(strategy="median")),
            ("modelo", ExtraTreesClassifier(
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1,
            )),
        ]),

        "parametros": {
            "modelo__n_estimators": [200, 300, 500, 700],
            "modelo__max_depth": [None, 10, 20, 30, 50],
            "modelo__min_samples_split": [2, 5, 10, 20],
            "modelo__min_samples_leaf": [1, 2, 4, 8],
            "modelo__max_features": [
                "sqrt",
                "log2",
                0.5,
                0.75,
                None,
            ],
            "modelo__bootstrap": [False, True],
        },
    },

    "Random Forest": {
        "pipeline": Pipeline([
            ("imputacion", SimpleImputer(strategy="median")),
            ("modelo", RandomForestClassifier(
                class_weight="balanced_subsample",
                random_state=SEED,
                n_jobs=-1,
            )),
        ]),

        "parametros": {
            "modelo__n_estimators": [200, 300, 500, 700],
            "modelo__max_depth": [None, 10, 20, 30, 50],
            "modelo__min_samples_split": [2, 5, 10, 20],
            "modelo__min_samples_leaf": [1, 2, 4, 8],
            "modelo__max_features": [
                "sqrt",
                "log2",
                0.5,
                0.75,
            ],
            "modelo__bootstrap": [True, False],
        },
    },
}

In [9]:
N_SPLITS_OUTER = 5
N_SPLITS_INNER = 4
N_ITER = 30

outer_cv = GroupKFold(n_splits=N_SPLITS_OUTER)

clases = np.sort(y.unique())

resultados_folds = []
predicciones_oof = {}
mejores_parametros = {}

for nombre, configuracion in modelos_busqueda.items():

    print(f"\n{'=' * 70}")
    print(nombre)
    print(f"{'=' * 70}")

    oof = np.empty(len(y), dtype=object)
    evaluada = np.zeros(len(y), dtype=bool)
    parametros_folds = []

    for fold, (idx_train, idx_test) in enumerate(
        outer_cv.split(X, y, groups),
        start=1,
    ):
        X_train = X[idx_train]
        X_test = X[idx_test]

        y_train = y[idx_train]
        y_test = y[idx_test]

        groups_train = groups[idx_train]
        groups_test = groups[idx_test]

        # Comprobación explícita de ausencia de grupos compartidos
        assert set(groups_train).isdisjoint(set(groups_test))

        inner_cv = GroupKFold(n_splits=N_SPLITS_INNER)

        busqueda = RandomizedSearchCV(
            estimator=clone(configuracion["pipeline"]),
            param_distributions=configuracion["parametros"],
            n_iter=N_ITER,
            scoring="f1_macro",
            cv=inner_cv,
            random_state=SEED,
            n_jobs=-1,
            verbose=0,
            refit=True,
        )

        # Los grupos también se entregan a la CV interna
        busqueda.fit(
            X_train,
            y_train,
            groups=groups_train,
        )

        pred = busqueda.predict(X_test)

        oof[idx_test] = pred
        evaluada[idx_test] = True

        accuracy = accuracy_score(y_test, pred)
        f1_macro = f1_score(
            y_test,
            pred,
            average="macro",
            zero_division=0,
        )

        resultados_folds.append({
            "modelo": nombre,
            "fold": fold,
            "accuracy": accuracy,
            "f1_macro": f1_macro,
            "mejor_f1_interno": busqueda.best_score_,
        })

        parametros_folds.append(busqueda.best_params_)

        print(
            f"Fold {fold}: "
            f"accuracy={accuracy:.4f} | "
            f"F1 macro={f1_macro:.4f}"
        )
        print("Mejores parámetros:", busqueda.best_params_)

    assert evaluada.all()

    predicciones_oof[nombre] = oof
    mejores_parametros[nombre] = parametros_folds

    print(f"\nClassification report OOF — {nombre}")
    print(
        classification_report(
            y,
            oof,
            labels=clases,
            digits=4,
            zero_division=0,
        )
    )


HistGradientBoosting
Fold 1: accuracy=0.9668 | F1 macro=0.9647
Mejores parámetros: {'modelo__min_samples_leaf': 20, 'modelo__max_leaf_nodes': 31, 'modelo__max_iter': 400, 'modelo__max_depth': None, 'modelo__learning_rate': 0.1, 'modelo__l2_regularization': 0.5}
Fold 2: accuracy=0.9638 | F1 macro=0.9624
Mejores parámetros: {'modelo__min_samples_leaf': 20, 'modelo__max_leaf_nodes': 31, 'modelo__max_iter': 400, 'modelo__max_depth': None, 'modelo__learning_rate': 0.1, 'modelo__l2_regularization': 0.5}
Fold 3: accuracy=0.9734 | F1 macro=0.9731
Mejores parámetros: {'modelo__min_samples_leaf': 20, 'modelo__max_leaf_nodes': 31, 'modelo__max_iter': 400, 'modelo__max_depth': None, 'modelo__learning_rate': 0.1, 'modelo__l2_regularization': 0.5}
Fold 4: accuracy=0.9727 | F1 macro=0.9717
Mejores parámetros: {'modelo__min_samples_leaf': 20, 'modelo__max_leaf_nodes': 31, 'modelo__max_iter': 400, 'modelo__max_depth': None, 'modelo__learning_rate': 0.1, 'modelo__l2_regularization': 0.5}
Fold 5: accura

d:\ANACONDA\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Fold 3: accuracy=0.9703 | F1 macro=0.9696
Mejores parámetros: {'modelo__n_estimators': 500, 'modelo__min_samples_split': 2, 'modelo__min_samples_leaf': 1, 'modelo__max_features': 0.5, 'modelo__max_depth': None, 'modelo__bootstrap': False}
Fold 4: accuracy=0.9705 | F1 macro=0.9693
Mejores parámetros: {'modelo__n_estimators': 500, 'modelo__min_samples_split': 2, 'modelo__min_samples_leaf': 1, 'modelo__max_features': 0.5, 'modelo__max_depth': None, 'modelo__bootstrap': False}
Fold 5: accuracy=0.9650 | F1 macro=0.9642
Mejores parámetros: {'modelo__n_estimators': 500, 'modelo__min_samples_split': 2, 'modelo__min_samples_leaf': 1, 'modelo__max_features': 0.5, 'modelo__max_depth': None, 'modelo__bootstrap': False}

Classification report OOF — Extra Trees
              precision    recall  f1-score   support

           0     0.9918    0.9779    0.9848      1856
           1     0.9533    0.9145    0.9335      1696
          10     0.9457    0.9795    0.9623      2488
          11     0.8097  

KeyboardInterrupt: 

In [6]:
modelo_final = Pipeline([
    (
        "imputacion",
        SimpleImputer(strategy="median")
    ),
    (
        "modelo",
        HistGradientBoostingClassifier(
            min_samples_leaf=20,
            max_leaf_nodes=31,
            max_iter=400,
            max_depth=None,
            learning_rate=0.1,
            l2_regularization=0.5,
            random_state=SEED
        )
    )
])

modelo_final.fit(X, y)

carpeta_modelos = Path("modelos")
carpeta_modelos.mkdir(exist_ok=True)

ruta_modelo = carpeta_modelos / "histgradientboosting_rehab.joblib"

joblib.dump(modelo_final, ruta_modelo)

print(f"Modelo guardado correctamente en: {ruta_modelo}")

Modelo guardado correctamente en: modelos\histgradientboosting_rehab.joblib


## Optimización de hiperparámetros

Después de comparar los modelos iniciales, se realizó una optimización de
hiperparámetros para **HistGradientBoosting** y **Extra Trees**.

Para obtener una estimación realista de su capacidad de generalización se utilizó
**validación cruzada anidada con GroupKFold**. La división por grupos mantuvo
separados a los participantes entre entrenamiento y evaluación, evitando que
ventanas pertenecientes a una misma persona aparecieran en ambos conjuntos.

La validación anidada estuvo formada por:

- **5 folds externos**, utilizados para estimar el desempeño final.
- **4 folds internos**, utilizados para seleccionar los hiperparámetros.
- **F1 macro** como métrica principal de optimización, debido a que todas las
  clases tienen la misma importancia.
- Predicciones **out-of-fold (OOF)** para evaluar cada observación únicamente
  cuando no participó en el entrenamiento del modelo correspondiente.

### Resultados de la validación externa

| Modelo | Accuracy OOF | F1 macro OOF | F1 weighted OOF |
|:--|--:|--:|--:|
| **HistGradientBoosting** | **0.9692** | **0.9678** | **0.9695** |
| Extra Trees | 0.9643 | 0.9629 | 0.9647 |

HistGradientBoosting obtuvo el mejor resultado global y superó a Extra Trees en
los cinco folds externos. La diferencia fue de aproximadamente **0.49 puntos
porcentuales en accuracy** y **0.49 puntos porcentuales en F1 macro**.

Además, los resultados de HistGradientBoosting fueron consistentes entre los
folds:

| Fold | Accuracy | F1 macro |
|:--:|--:|--:|
| 1 | 0.9668 | 0.9647 |
| 2 | 0.9638 | 0.9624 |
| 3 | 0.9734 | 0.9731 |
| 4 | 0.9727 | 0.9717 |
| 5 | 0.9693 | 0.9681 |

### Hiperparámetros seleccionados

La siguiente configuración fue seleccionada en cuatro de los cinco folds
externos, lo que muestra estabilidad en la búsqueda:

```python
{
    "min_samples_leaf": 20,
    "max_leaf_nodes": 31,
    "max_iter": 400,
    "max_depth": None,
    "learning_rate": 0.1,
    "l2_regularization": 0.5,
}